# Generate answers (Colab)

[Open in Colab](https://colab.research.google.com/github/khacbac/nlp/blob/main/experiment/generate_answers.ipynb)

Answers are saved to **Google Drive** (`MyDrive/nlp-answers/answers_<model>.jsonl`), one line per answer,
so nothing is lost when Colab disconnects or a quota runs out.
**To continue after any error: run the cells again from the top.** Finished answers are skipped;
failed or empty ones are retried.

- Run cells 1-3 every time.
- Gemini and gpt-oss run on a CPU runtime. Qwen needs *Runtime > Change runtime type > T4 GPU*.
- Gemini's free tier allows about 500 requests a day: if it stops with "Quota exhausted", run it again tomorrow.
- Put the keys in Colab *Secrets* (key icon on the left): `GEMINI_API_KEY` (aistudio.google.com),
  `GROQ_API_KEY` (console.groq.com), optionally `OPENROUTER_API_KEY`. Turn on notebook access for each.
- Annotators must not open the answer files: they name the model of each answer.

## 1. Drive + code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = "/content/drive/MyDrive/nlp-answers"
REPO = "/content/nlp"

import os
os.makedirs(OUT_DIR, exist_ok=True)
# The Colab copy of the repo is throw-away (answers live on Drive), so always reset it to the latest main.
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} fetch -q origin && git -C {REPO} reset -q --hard origin/main
else:
    !git clone -q https://github.com/khacbac/nlp.git {REPO}
%cd {REPO}/experiment
!git log -1 --format="code at %h: %s"

## 2. API keys

In [ ]:
from google.colab import userdata

for name in ["GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY"]:
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: set")
    except Exception:
        print(f"{name}: not found in Secrets (only needed for the model that uses it)")

## 3. Progress

In [ ]:
import json
from pathlib import Path

def progress():
    qids = {json.loads(l)["qid"] for l in open("data/questions.jsonl", encoding="utf-8")}
    for m in ["qwen", "gpt-oss", "gemini"]:
        path = Path(OUT_DIR) / f"answers_{m}.jsonl"
        done = set()
        if path.exists():
            for line in path.open(encoding="utf-8"):
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if row.get("answer"):
                    done.add(row["qid"])
        print(f"{m:8s} {len(done & qids):4d} / {len(qids)}")

progress()

## 4. Trial: 3 Gemini answers
Read them: Vietnamese, 2-4 sentences, plain prose, faithful to the passage?

In [ ]:
!python generate_answers.py --model gemini --limit 3 --out-dir {OUT_DIR}

for line in open(f"{OUT_DIR}/answers_gemini.jsonl", encoding="utf-8").readlines()[:3]:
    row = json.loads(line)
    print(row["qid"], f'({row["num_words"]} words, {row["finish_reason"]})')
    print(row["answer"], "\n")

## 5. Gemini Flash-Lite (CPU runtime, ~40 min per 500 requests; may take 2 days)

In [ ]:
!python generate_answers.py --model gemini --out-dir {OUT_DIR}
progress()

## 6. gpt-oss-20b (CPU runtime, ~40 min)
If Groq's free quota runs out, set `PROVIDER = "openrouter"` and rerun.

In [ ]:
PROVIDER = "groq"
!python generate_answers.py --model gpt-oss --provider {PROVIDER} --out-dir {OUT_DIR}
progress()

## 7. Qwen2.5-7B-Instruct (T4 GPU runtime, ~10 min + ~5 min install)

In [ ]:
import subprocess
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then run from cell 1 again.")
!pip install -q vllm
!python generate_answers.py --model qwen --out-dir {OUT_DIR}
progress()

## Next

When all three models show 500 / 500, the person who builds the grading sheets downloads the three
`answers_*.jsonl` files from `MyDrive/nlp-answers` into `experiment/data/` and runs
`python make_grading_sheet.py` (see README).